In [9]:
import pandas as pd
from rdflib import Graph, Namespace, Literal, RDF, XSD

In [10]:
rent_data = pd.DataFrame([
    {"district": "Connewitz", "offer_rent_per_sqm": 13.20, "utilities": 120},
    {"district": "Plagwitz", "offer_rent_per_sqm": 14.10, "utilities": 120},
    {"district": "Gruenau", "offer_rent_per_sqm": 8.40, "utilities": 120},
])

groups = pd.DataFrame([
    {"group": "Students", "monthly_income": 992},
    {"group": "Apprentices", "monthly_income": 900},
    {"group": "CitizenBenefitRecipients", "monthly_income": 563},
    {"group": "MedianEarners", "monthly_income": 1877},
])

STANDARD_SIZE_SQM = 30
AFFORDABLE_THRESHOLD = 0.30
CRITICAL_THRESHOLD = 0.45

rent_data

,district,offer_rent_per_sqm,utilities
0,Connewitz,13.2,120
1,Plagwitz,14.1,120
2,Gruenau,8.4,120


In [11]:
records = []

for _, district in rent_data.iterrows():
    warm_rent = district["offer_rent_per_sqm"] * STANDARD_SIZE_SQM + district["utilities"]

    for _, group in groups.iterrows():
        stress_score = warm_rent / group["monthly_income"]

        if stress_score <= AFFORDABLE_THRESHOLD:
            status = "affordable"
        elif stress_score <= CRITICAL_THRESHOLD:
            status = "critical"
        else:
            status = "not_affordable"

        records.append({
            "district": district["district"],
            "group": group["group"],
            "warm_rent_30sqm": round(warm_rent, 2),
            "monthly_income": group["monthly_income"],
            "stress_score": round(stress_score, 3),
            "status": status
        })

affordability = pd.DataFrame(records)
affordability

,district,group,warm_rent_30sqm,monthly_income,stress_score,status
0,Connewitz,Students,516.0,992,0.520,not_affordable
1,Connewitz,Apprentices,516.0,900,0.573,not_affordable
2,Connewitz,CitizenBenefitRecipients,516.0,563,0.917,not_affordable
3,Connewitz,MedianEarners,516.0,1877,0.275,affordable
4,Plagwitz,Students,543.0,992,0.547,not_affordable
5,Plagwitz,Apprentices,543.0,900,0.603,not_affordable
6,Plagwitz,CitizenBenefitRecipients,543.0,563,0.964,not_affordable
7,Plagwitz,MedianEarners,543.0,1877,0.289,affordable
8,Gruenau,Students,372.0,992,0.375,critical
9,Gruenau,Apprentices,372.0,900,0.413,critical


In [12]:
EX = Namespace("https://example.org/leipzig-housing/")
g = Graph()
g.bind("lh", EX)

for _, row in affordability.iterrows():
    district_uri = EX[row["district"]]
    group_uri = EX[row["group"]]
    observation_uri = EX[f"{row['district']}_{row['group']}_Affordability"]

    g.add((district_uri, RDF.type, EX.District))
    g.add((group_uri, RDF.type, EX.SocialGroup))
    g.add((observation_uri, RDF.type, EX.AffordabilityObservation))

    g.add((observation_uri, EX.forDistrict, district_uri))
    g.add((observation_uri, EX.forGroup, group_uri))
    g.add((observation_uri, EX.hasWarmRent30sqm, Literal(row["warm_rent_30sqm"], datatype=XSD.decimal)))
    g.add((observation_uri, EX.hasMonthlyIncome, Literal(row["monthly_income"], datatype=XSD.decimal)))
    g.add((observation_uri, EX.hasHousingStressScore, Literal(row["stress_score"], datatype=XSD.decimal)))
    g.add((observation_uri, EX.hasAffordabilityStatus, Literal(row["status"])))

print(f"Created {len(g)} RDF triples.")

Created 91 RDF triples.


In [13]:
g.serialize("../rdf_output/housing_affordability_dummy.ttl", format="turtle")

print("RDF file saved to rdf_output/housing_affordability_dummy.ttl")

RDF file saved to rdf_output/housing_affordability_dummy.ttl


In [14]:
print(g.serialize(format="turtle")[:2000])

@prefix lh: <https://example.org/leipzig-housing/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

lh:Connewitz_Apprentices_Affordability a lh:AffordabilityObservation ;
    lh:forDistrict lh:Connewitz ;
    lh:forGroup lh:Apprentices ;
    lh:hasAffordabilityStatus "not_affordable" ;
    lh:hasHousingStressScore 0.573 ;
    lh:hasMonthlyIncome 900.0 ;
    lh:hasWarmRent30sqm 516.0 .

lh:Connewitz_CitizenBenefitRecipients_Affordability a lh:AffordabilityObservation ;
    lh:forDistrict lh:Connewitz ;
    lh:forGroup lh:CitizenBenefitRecipients ;
    lh:hasAffordabilityStatus "not_affordable" ;
    lh:hasHousingStressScore 0.917 ;
    lh:hasMonthlyIncome 563.0 ;
    lh:hasWarmRent30sqm 516.0 .

lh:Connewitz_MedianEarners_Affordability a lh:AffordabilityObservation ;
    lh:forDistrict lh:Connewitz ;
    lh:forGroup lh:MedianEarners ;
    lh:hasAffordabilityStatus "affordable" ;
    lh:hasHousingStressScore 0.275 ;
    lh:hasMonthlyIncome 1877.0 ;
    lh:hasWarmRent30sqm 516.0 .

l

In [15]:
query = """
PREFIX lh: <https://example.org/leipzig-housing/>

SELECT ?district ?group ?stress ?status
WHERE {
    ?obs a lh:AffordabilityObservation ;
         lh:forDistrict ?district ;
         lh:forGroup ?group ;
         lh:hasHousingStressScore ?stress ;
         lh:hasAffordabilityStatus ?status .

    FILTER(?status = "not_affordable")
}
"""

for row in g.query(query):
    print(row)

(rdflib.term.URIRef('https://example.org/leipzig-housing/Connewitz'), rdflib.term.URIRef('https://example.org/leipzig-housing/Students'), rdflib.term.Literal('0.52', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')), rdflib.term.Literal('not_affordable'))
(rdflib.term.URIRef('https://example.org/leipzig-housing/Connewitz'), rdflib.term.URIRef('https://example.org/leipzig-housing/Apprentices'), rdflib.term.Literal('0.573', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')), rdflib.term.Literal('not_affordable'))
(rdflib.term.URIRef('https://example.org/leipzig-housing/Connewitz'), rdflib.term.URIRef('https://example.org/leipzig-housing/CitizenBenefitRecipients'), rdflib.term.Literal('0.917', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#decimal')), rdflib.term.Literal('not_affordable'))
(rdflib.term.URIRef('https://example.org/leipzig-housing/Plagwitz'), rdflib.term.URIRef('https://example.org/leipzig-housing/Students'), rdfli

In [16]:
query_results = []

for row in g.query(query):
    district = str(row.district).split("/")[-1]
    group = str(row.group).split("/")[-1]
    stress = float(row.stress)
    status = str(row.status)

    query_results.append({
        "district": district,
        "group": group,
        "stress_score": stress,
        "status": status
    })

not_affordable_df = pd.DataFrame(query_results)
not_affordable_df

,district,group,stress_score,status
0,Connewitz,Students,0.520,not_affordable
1,Connewitz,Apprentices,0.573,not_affordable
2,Connewitz,CitizenBenefitRecipients,0.917,not_affordable
3,Plagwitz,Students,0.547,not_affordable
4,Plagwitz,Apprentices,0.603,not_affordable
5,Plagwitz,CitizenBenefitRecipients,0.964,not_affordable
6,Gruenau,CitizenBenefitRecipients,0.661,not_affordable
